

This notebook provides a comprehensive workflow for building and evaluating LLMs designed to classify scientific paper reviews as 'Accept' or 'Reject'. The process involves several key stages:

1.  **Sentence Embedding and Summarization**: It begins by processing raw labeled sentences from scientific papers. Sentence embeddings are generated using `MiniLM-L6-v2 SBERT`, and a label-aware, weighted summarization technique is applied. This involves clustering sentences with KMeans and selecting representatives, followed by optional paraphrasing using `T5-small` to create concise summaries per paper. The output is saved to `multi_paper_summaries_weighted.json`.

2.  **Data Preparation for Classification**: The generated summaries are then processed to convert paper decisions (e.g., 'accept', 'reject') into binary labels (1 for accept, 0 for reject). Per-label sentences are aggregated into a single text block for each paper, and the data is saved in `processed_reviews.jsonl`.

3.  **Model Benchmarking**: Several pre-trained language models are benchmarked for their zero-shot classification performance on the processed data:
    *   `DeBERTa-v3-Base`
    *   `Mistral-7B` (4-bit quantized)
    *   `Phi-3 Mini` (4-bit quantized)
    
    Metrics such as Accuracy, F1-score, Precision, Recall, and Confusion Matrices are reported, along with performance metrics like latency and memory usage.

4.  **Dataset Splitting**: The `processed_reviews.jsonl` dataset is split into `train.jsonl` (80%), `val.jsonl` (10%), and `test.jsonl` (10%) using stratified sampling to ensure label distribution is maintained across splits.

5.  **Model Fine-tuning**: The notebook then proceeds to fine-tune the following models on the prepared datasets for improved classification performance:
    *   `Mistral-7B` (using 4-bit quantization and LoRA)
    *   `DeBERTa-v3` (with weighted loss for class imbalance)
    *   `Phi-3 Mini` (using LoRA)

    Detailed evaluation metrics including Accuracy, Precision, Recall, F1-score, and ROC AUC are provided for train, validation, and test sets, along with confusion matrices and performance statistics post-fine-tuning. Error analysis with sample false positives and false negatives is also included to understand model behavior.

This notebook serves as a complete pipeline from raw text processing to fine-tuned model evaluation for review decision classification.

##Sentence Embedding

This script builds label-aware, weighted summaries for thousands of papers.
It first loads labeled sentences and computes MiniLM-L6-v2 SBERT embeddings for each sentence.
For every label within a paper (ex. strengths, weaknesses, suggestions), it uses a weight table to decide how many representative sentences to extract. Sentences are grouped using KMeans clustering, and from each cluster the sentence closest to the centroid is selected as a top representative.
All selected sentences across labels are paraphrased using T5-small to produce a concise, unified summary text (can be turned off using paraphrase = false).
Paper-level metadata (scores, decisions, titles) is merged, and each paper receives its per-label representative sentences and (optionally) a paraphrased overall summary


Everything is saved into `multi_paper_summaries_weighted.json`, along with timing and GPU memory usage.

In [ ]:
import os
from collections import defaultdict
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch
import pandas as pd
import time
import json



# configuration
input_file = "/content/filtered_train_labeled_sentences.txt"
meta_file = "/content/full_data_info (1).txt"
output_file = "multi_paper_summaries_weighted.json"



clusters_per_label = 2
paraphrase = True
device = "cuda" if torch.cuda.is_available() else "cpu"
random_seed = 42
num_papers_to_summarize = 5000


importance_table = pd.DataFrame({
    "label": ["weakness", "strength", "abstract", "suggestion", "misc",
              "rebuttal_process", "rating_summary", "ac_disagreement"],
    "importance_hybrid_norm": [0.129304, 0.126604, 0.05, 0.0,
                               0, 0.123941, 0.123938, 0.123938]
    #importance score per label --> what is included more or less in the overall summary
})

label_weights = dict(zip(importance_table["label"], importance_table["importance_hybrid_norm"]))

#  labeled sentences into dict per paper
papers = defaultdict(lambda: defaultdict(list))
skipped_lines = 0
with open(input_file, 'r', encoding='utf-8') as f:
    for line in f:



        line = line.strip()
        if not line:
            continue
        parts = line.split('\t')
        if len(parts) != 3:


            skipped_lines += 1
            continue
        paper_id, sentence, label = parts
        papers[paper_id][label].append(sentence)

print(f"loaded {len(papers)} papers. skipped {skipped_lines} invalid lines.")

# load sentence transformer model
sbert_model = SentenceTransformer('all-MiniLM-L6-v2')

# load t5 model for paraphrasing if enabled
if paraphrase:
    tokenizer = AutoTokenizer.from_pretrained("t5-small")
    t5_model = AutoModelForSeq2SeqLM.from_pretrained("t5-small").to(device)

# function to paraphrase text
def paraphrase_text(text, max_input_length=512, max_output_length=80, min_output_length=20):
    inputs = tokenizer("summarize: " + text,
                       return_tensors="pt",
                       max_length=max_input_length,
                       truncation=True).to(device)
    summary_ids = t5_model.generate(**inputs,
                                    max_length=max_output_length,
                                    min_length=min_output_length)
    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)

# read meta information for papers
meta_cols = ["dataset", "paper_id", "year", "paper_ref", "review_scores",
             "avg_score", "decision", "title", "url", "n_reviews",
             "all_labels", "unique_labels"]
df_meta = pd.read_csv(meta_file, sep="\t", names=meta_cols, header=None)

meta_dict = {}
for paper_id, group in df_meta.groupby("paper_id"):
    meta_dict[paper_id] = {
        "review_scores": "; ".join(group["review_scores"].astype(str)),
        "avg_score": "; ".join(group["avg_score"].astype(str)),
        "decision": "; ".join(group["decision"].astype(str)),
        "title": group["title"].iloc[0]
    }

# summarize papers by label
all_summaries = {}
start_time = time.time()

paper_ids = list(papers.keys())[:num_papers_to_summarize]
print(f"\nprocessing {len(paper_ids)} papers")

for paper_id in paper_ids:
    labels_dict = papers[paper_id]
    label_sentences = {}
    all_top_sentences = []

    for label, sentences in labels_dict.items():
        if not sentences:
            continue

        embeddings = sbert_model.encode(sentences)
        weight = label_weights.get(label, 1.0)
        n_top = max(1, int(clusters_per_label * weight * 10))

        n_clusters = min(n_top, len(sentences))

        if n_clusters == 1:
            top_sentences = [sentences[0]]
        else:
            kmeans = KMeans(n_clusters=n_clusters, random_state=random_seed)
            kmeans.fit(embeddings)
            centers = kmeans.cluster_centers_
            top_sentences = []

            for center in centers:
                sims = np.dot(embeddings, center) / (np.linalg.norm(embeddings, axis=1) * np.linalg.norm(center))
                top_idx = np.argmax(sims)
                top_sentences.append(sentences[top_idx])

        label_sentences[label] = top_sentences
        all_top_sentences.extend(top_sentences)

    if paraphrase and all_top_sentences:
        summary_text = paraphrase_text(" ".join(all_top_sentences))
        all_summaries[paper_id] = {"summary": summary_text, "per_label": label_sentences}
    else:
        all_summaries[paper_id] = {"per_label": label_sentences}

elapsed_time = time.time() - start_time
gpu_mem_used = torch.cuda.max_memory_allocated() / (1024**2) if device == "cuda" else None

# combine summaries with meta information
output_list = []
for paper_id in paper_ids:
    paper_meta = meta_dict.get(paper_id, {})
    output_list.append({
        "paper_id": paper_id,

        "review_scores": paper_meta.get("review_scores", ""),
        "avg_score": paper_meta.get("avg_score", ""),
        "decision": paper_meta.get("decision", ""),


        "per_label_sentences": all_summaries[paper_id].get("per_label", {})
    })

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(output_list, f, indent=4, ensure_ascii=False)

print(f"summaries for {len(paper_ids)} papers saved to {output_file}")
print(f"computation time: {elapsed_time:.2f} sec")
if gpu_mem_used:
    print(f"gpu memory used: {gpu_mem_used:.2f} mb")


for pid in paper_ids:
    print(f"paper id: {pid}")



##Multi label binarization + processing embedding output

This script processes paper summaries from `multi_paper_summaries_weighted.json` to prepare them for model benchmarking and finetuning.


- Decision labeling – converts the paper decision text to a binary label: `1` for "accept" and `0` for "reject".

- Sentence aggregation – combines all per-label sentences for each paper into a single text string.

- JSONL output – saves the processed data in `processed_reviews.jsonl`, with each entry containing:
  - `paper_id`
  - `text` (combined sentences)
  - `label` (binary decision)

This format is for training or evaluating.

In [ ]:
import json
import re


with open("/content/multi_paper_summaries_weighted.json", "r") as f:
    data = json.load(f)

processed = []


# normalize --> accept is 1, reject is 0

def decision_to_label(decision_text):
    decision_text = decision_text.lower()
    if "accept" in decision_text:
        return 1
    return 0

#combining all per label sentences
for entry in data:

    paper_id = entry["paper_id"]
    decision_text = entry["decision"]
    label = decision_to_label(decision_text)


    combined_sentences = []
    for section_name, sentences in entry["per_label_sentences"].items():
        for s in sentences:
            s = s.strip()


            if len(s) > 0:
                combined_sentences.append(s)


    full_text = " ".join(combined_sentences)

    processed.append({
        "paper_id": paper_id,
        "text": full_text,
        "label": label
    })

#create processed reviews jsonl
with open("processed_reviews.jsonl", "w") as f:
    for item in processed:
        f.write(json.dumps(item) + "\n")

print("processed_reviews.jsonl")


# BENCHMARK: *DeBERTa-v3-Base*


**Model**: `microsoft/deberta-v3-base` fine-tuned for 2 classes.

**Pipeline**:

1. Load and preprocess dataset (`.jsonl`) into a HuggingFace Dataset.

2. Tokenize reviews with truncation (max_length=512) and padding.

3. Batch inference using DataLoader (batch size = 32).

4. Compute predictions with torch.argmax on model logits (accept/reject classes).

**Metrics**:

- Accuracy: 0.362
- F1: 0.531
- Precision: 0.362
- Recall: 1.000

| True \ Pred | Pred Reject (0) | Pred Accept (1) |
|------------|----------------|----------------|
| Reject (0) | 0              | 3192           |
| Accept (1) | 0              | 1808           |


In [ ]:

!pip install transformers datasets evaluate accelerate torch -q
!pip install psutil scikit-learn -q


import json, pandas as pd, torch, time, psutil
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import confusion_matrix, precision_score, recall_score
from torch.utils.data import DataLoader
import evaluate


jsonl_path = "/content/processed_reviews.jsonl"
rows = [json.loads(line) for line in open(jsonl_path)]
df = pd.DataFrame(rows)
print(f"Total samples: {len(df)}")

# using only review text
df["combined"] = df["text"]
dataset = Dataset.from_pandas(df)

# load model and tokenizer
model_name = "microsoft/deberta-v3-base"
tokenizer = "microsoft"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device).eval()

# tokenize
dataset = dataset.map(lambda b: tokenizer(b["combined"], truncation=True, max_length=512, padding="max_length"), batched=True)
dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

# inference
loader = DataLoader(dataset, batch_size=32)
all_preds, all_labels = [], []
start_time = time.time()
for batch in loader:
    input_ids, attention_mask, labels = batch["input_ids"].to(device), batch["attention_mask"].to(device), batch["label"].to(device)
    with torch.no_grad():
        preds = torch.argmax(model(input_ids=input_ids, attention_mask=attention_mask).logits, dim=-1)
    all_preds.extend(preds.cpu().tolist())
    all_labels.extend(labels.cpu().tolist())
end_time = time.time()

# metrics
accuracy = evaluate.load("accuracy").compute(predictions=all_preds, references=all_labels)["accuracy"]
f1 = evaluate.load("f1").compute(predictions=all_preds, references=all_labels)["f1"]
precision = precision_score(all_labels, all_preds, zero_division=0)
recall = recall_score(all_labels, all_preds, zero_division=0)
conf_mat = confusion_matrix(all_labels, all_preds)
conf_df = pd.DataFrame(conf_mat, index=["Reject (0)","Accept (1)"], columns=["Pred Reject (0)","Pred Accept (1)"])

latency = (end_time - start_time) / len(dataset)
peak_mem = psutil.Process().memory_info().peak_wset if hasattr(psutil.Process().memory_info(), 'peak_wset') else psutil.Process().memory_info().rss


print(f"Total samples evaluated: {len(dataset)}")
print(f"Accuracy: {accuracy:.4f}, F1: {f1:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}")
print("Confusion matrix (rows: true, columns: predicted):")
print(conf_df)
print(f"Inference latency per sample: {latency*1000:.2f} ms")
print(f"Peak memory usage: {peak_mem / (1024**2):.2f} MB")


# BENCHMARK: *Mistral-7B 4-bit*

**Model**: `'mistralai/Mistral-7B-v0.1'` with 4-bit quantization (`nf4`, double quantization).

**Pipeline**:

1. Load and preprocess dataset (`.jsonl`) and combine review text.
2. Prepare tokenizer and model with 4-bit quantization for memory efficiency.
3. Give prompt.
4. Compute predictions using logits comparison between `'Accept'` and `'Reject'` tokens.
5. Batch size = 1 to avoid OOM and handle variable-length inputs safely.

**Metrics**:

- Accuracy: 0.637
- F1: 0.366
- Precision: 0.496
- Recall: 0.289

| True \ Pred | Pred Reject (0) | Pred Accept (1) |
|------------|----------------|----------------|
| Reject (0) | 2661           | 531            |
| Accept (1) | 1285           | 523            |

**Performance**:

- Latency per sample: 229.95 ms
- Peak GPU memory: ~6780 MB
- System RAM: ~1775 MB


In [ ]:

!pip install --upgrade transformers accelerate bitsandbytes tqdm -q


import json
import pandas as pd
import torch
import time
import psutil
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix


jsonl_path = "/content/processed_reviews (2).jsonl"
rows = [json.loads(line) for line in open(jsonl_path)]
df = pd.DataFrame(rows)
df["combined"] = df["text"]


print(f"Total samples: {len(df)}")



model_name = "mistralai/Mistral-7B-v0.1"




bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, # using 4-bit precision instead of FP16/32
    bnb_4bit_use_double_quant=True, # first quantize weights to 4-bit / then apply a second small scaling step to reduce quantization error
    bnb_4bit_quant_type="nf4", # using normalfloat-4 (as our 4 bit type)
    bnb_4bit_compute_dtype=torch.float16, # normal computations are still done in float16, but weights are stored in 40bit
)

tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token # add a padding token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    attn_implementation="sdpa", # scaled dot-product attention (standard implementation)
)
model.eval()

#logit classifier
def classify_logits(batch_texts):
    inputs = tokenizer( #convert combined text into token IDs (truncate longer ones to 256 tokens)
        batch_texts,
        padding=True,
        truncation=True,
        max_length=256,
        return_tensors="pt"
    ).to(model.device)

#disable gradient calculation as we only want predictions
    with torch.no_grad():
        outputs = model(**inputs)

#for classification, only the last token's logits are used (model predicts the decision last so we pick the token at the end)
    logits = outputs.logits[:, -1, :]


#accept (1) and reject(0)
    acc_id = tokenizer("Accept", add_special_tokens=False).input_ids[0]
    rej_id = tokenizer("Reject", add_special_tokens=False).input_ids[0]


#compare which logit is greater in [logit accept, logit reject] --> pick the greater one
    preds = (logits[:, acc_id] > logits[:, rej_id]).long()

    # clear cache immediately
    del inputs, outputs, logits
    torch.cuda.empty_cache()

    return preds.cpu().tolist()

#inference
batch_size = 1  #  prevent OOM (as this is a larger model, significant memory is used in 4 bit mode, so processing one sample at a time is safe)
all_preds = []
start_time = time.time()

for i in tqdm(range(0, len(df), batch_size), desc="Classifying", ncols=80):
    batch_reviews = df["combined"].iloc[i:i+batch_size].tolist()
    prompts = [f"Review: {r[:400]}\nDecision:" for r in batch_reviews]  # PROMPT
    preds = classify_logits(prompts)
    all_preds.extend(preds)

end_time = time.time()

#metrics
true_labels = df["label"].tolist()


accuracy = accuracy_score(true_labels, all_preds)
f1 = f1_score(true_labels, all_preds)
precision = precision_score(true_labels, all_preds)

recall = recall_score(true_labels, all_preds)
conf_mat = confusion_matrix(true_labels, all_preds)

print(f"\nAccuracy: {accuracy:.4f}")
print(f"F1: {f1:.4f}  Precision: {precision:.4f}  Recall: {recall:.4f}\n")
print(pd.DataFrame(
    conf_mat,
    index=["Reject (0)", "Accept (1)"],
    columns=["Pred Reject (0)", "Pred Accept (1)"]
))


latency = (end_time - start_time) / len(df)
peak_gpu = torch.cuda.max_memory_allocated() / 1024**2
ram = psutil.Process().memory_info().rss / 1024**2


print(f"\nLatency per sample: {latency*1000:.2f} ms")
print(f"Peak GPU memory: {peak_gpu:.2f} MB")
print(f"System RAM: {ram:.2f} MB")


#BENCHMARK: *Phi-3 Mini*

**Model**: `'phi-3-mini-4k-instruct'` with 4-bit quantization (`nf4`).

Accuracy: 0.3802, F1: 0.5124
Precision: 0.3580, Recall: 0.9004

Confusion Matrix:

| Actual \ Pred | Pred Reject (0) | Pred Accept (1) |
|--------------|-----------------|------------------|
| Reject (0)   | 273             | 2919             |
| Accept (1)   | 180             | 1628             |

Latency per sample: 135.63 ms
___
Samples per second: 7.37
___
Peak RAM used: 2534.47 MB
___

In [ ]:
import json, pandas as pd, torch, time, psutil
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, BitsAndBytesConfig
from sklearn.metrics import confusion_matrix, precision_score, recall_score
from torch.utils.data import DataLoader
import evaluate


jsonl_path = "/content/processed_reviews.jsonl"
rows = [json.loads(line) for line in open(jsonl_path)]
df = pd.DataFrame(rows)
print(f"Total samples: {len(df)}")

df["combined"] = df["text"]
dataset = Dataset.from_pandas(df)


model_name = "microsoft/phi-3-mini-4k-instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.padding_side = "left"
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    quantization_config=bnb_config,
    device_map="auto"
)

model.eval()


def tok(batch):
    return tokenizer(
        batch["combined"],
        truncation=True,
        max_length=256,
        padding="max_length"
    )

dataset = dataset.map(tok, batched=True)
dataset.set_format(type="torch",
                   columns=["input_ids", "attention_mask", "label"])


loader = DataLoader(dataset, batch_size=4)

all_preds, all_labels = [], []
start_time = time.time()

for batch in loader:
    with torch.no_grad():
        logits = model(
            input_ids=batch["input_ids"].to(model.device),
            attention_mask=batch["attention_mask"].to(model.device)
        ).logits
        preds = torch.argmax(logits, dim=-1)

    all_preds.extend(preds.cpu().tolist())
    all_labels.extend(batch["label"].tolist())

end_time = time.time()


accuracy = evaluate.load("accuracy").compute(
    predictions=all_preds, references=all_labels)["accuracy"]

f1 = evaluate.load("f1").compute(
    predictions=all_preds, references=all_labels)["f1"]

precision = precision_score(all_labels, all_preds, zero_division=0)
recall = recall_score(all_labels, all_preds, zero_division=0)

conf_mat = confusion_matrix(all_labels, all_preds)
conf_df = pd.DataFrame(conf_mat,
    index=["Reject (0)", "Accept (1)"],
    columns=["Pred Reject (0)", "Pred Accept (1)"]
)

latency = (end_time - start_time) / len(dataset)
total_time = end_time - start_time
samples_per_sec = len(dataset) / total_time
peak_ram_mb = psutil.Process().memory_info().rss / (1024**2)

print(f"Total samples evaluated: {len(dataset)}")
print(f"Accuracy: {accuracy:.4f}, F1: {f1:.4f}")
print(f"Precision: {precision:.4f}, Recall: {recall:.4f}")

print("\nConfusion Matrix:")
print(conf_df)

print(f"\nLatency per sample: {latency*1000:.2f} ms")
print(f"Samples per second: {samples_per_sec:.2f}")
print(f"Peak RAM used: {peak_ram_mb:.2f} MB")


#Splitting `processed_reviews` into `train.jsonl` (80%), `validation.jsonl` (10%),and `test.jsonl` (10%) for finetuning

- Every dataset is equally stratified and randomized

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
import json


data = []
with open("/content/processed_reviews.jsonl", "r") as f:
    for line in f:
        data.append(json.loads(line))

df = pd.DataFrame(data)


train_val, test = train_test_split(
    df,
    test_size=0.10,
    random_state=42,
    stratify=df["label"],
    shuffle=True
)

# validation is 10% of original = 10/90 ~ 0.1111 of train_val
train, val = train_test_split(
    train_val,
    test_size=0.1111,
    random_state=42,
    stratify=train_val["label"],
    shuffle=True
)

print("Train size:", len(train), "\nLabel distribution:\n", train["label"].value_counts(normalize=True))
print("Val size:", len(val), "\nLabel distribution:\n", val["label"].value_counts(normalize=True))
print("Test size:", len(test), "\nLabel distribution:\n", test["label"].value_counts(normalize=True))


train.to_json("train.jsonl", orient="records", lines=True, index=False)
val.to_json("val.jsonl", orient="records", lines=True, index=False)
test.to_json("test.jsonl", orient="records", lines=True, index=False)


# FINETUNE: Mistral-7B (4-bit + LoRA)

##Pipeline

1. *Load + Quantize Model*
   - Load Mistral-7B in 4-bit (NF4) with BitsAndBytes.
   - Enable gradient checkpointing and prepare for k-bit training.
   - Apply LoRA (r=4, α=16) to attention projections.

2. *Train + Evaluate*
   - Tokenize dataset with fixed prompt format and max length 96.
   - Train for 1 epoch with batch size 1 and gradient accumulation 4.
   - Evaluate on validation + test sets using logits from the last token.

3. *Inference + Metrics*
   - Run manual test loop measuring per-sample latency.
   - Compute confusion matrix, accuracy, F1, precision, recall.
   - Log GPU/CPU memory usage and save fine-tuned model.


##Eval metrics
| Metric                   | Value   |
|--------------------------|---------|
| eval_loss                | 1.821   |
| eval_accuracy            | 0.676   |
| eval_f1                  | 0.503   |
| eval_precision           | 0.587   |
| eval_recall              | 0.441   |
| eval_runtime             | 52.4    |
| eval_samples_per_second  | 87.1    |

## Test Metrics

| Metric                   | Value    |
|--------------------------|----------|
| test_loss                | 1.784    |
| test_accuracy            | 0.684    |
| test_f1                  | 0.512    |
| test_precision           | 0.598    |
| test_recall              | 0.449    |
| test_runtime             | 63.9     |
| test_samples_per_second  | 71.4     |

---

## Confusion Matrix

| True \ Pred | Pred Reject (0) | Pred Accept (1) |
|-------------|-----------------|-----------------|
| Reject (0)  | 2795            | 402             |
| Accept (1)  | 1048            | 741             |

---

## Performance Summary

| Metric                 | Value       |
|------------------------|-------------|
| Test Accuracy          | 0.6843      |
| Avg Latency (ms)       | 214.72      |
| Peak GPU Memory (MB)   | 7020.41     |
| Final RAM Usage (MB)   | 1860.55     |


In [ ]:

# !pip install -q bitsandbytes==0.43.1
# !pip install -q transformers==4.43.3 accelerate==0.31.0 peft==0.11.1 datasets

import os
import json
import time
import psutil
import numpy as np
from tqdm import tqdm
import torch
import gc

# prevent CUDA fragmentation and memory splitting
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:128"

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
    EarlyStoppingCallback,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix



#paths and model config
MODEL_NAME = "mistralai/Mistral-7B-v0.1"
TRAIN_FILE = "/content/train.jsonl"
VAL_FILE = "/content/val.jsonl"
TEST_FILE = "/content/test.jsonl"
OUTPUT_DIR = "./mistral_finetuned"

# training / memory parameters
MAX_LENGTH = 96
PER_DEVICE_BATCH = 1
GRADIENT_ACCUMULATION = 4
EPOCHS = 1
LEARNING_RATE = 5e-5
WEIGHT_DECAY = 0.1
WARMUP_RATIO = 0.06
REPORT_TO = "none"
LOGGING_STEPS = 50

# LoRA parameters
LORA_R = 4
LORA_ALPHA = 16
LORA_DROPOUT = 0.1


# memory utilities

def clear_memory():
    gc.collect()
    torch.cuda.empty_cache()
    if torch.cuda.is_available():
        torch.cuda.synchronize()

def get_ram():
    return psutil.Process().memory_info().rss / 1024**2

def get_gpu_mem():
    return (torch.cuda.memory_allocated() / 1024**2) if torch.cuda.is_available() else 0

def print_memory_stats():
    print(f"RAM: {get_ram():.2f} MB | GPU: {get_gpu_mem():.2f} MB")



# load dataset (manual JSONL loader to avoid RAM spikes)

def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(l) for l in f]

print("Loading datasets")
train_data = load_jsonl(TRAIN_FILE)
val_data = load_jsonl(VAL_FILE)
test_data = load_jsonl(TEST_FILE)

print(f"Train samples: {len(train_data)}")
print(f"Val samples:   {len(val_data)}")
print(f"Test samples:  {len(test_data)}")
print_memory_stats()



# load tokenizer & 4-bit Mistral model

clear_memory()

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token  # required for padding

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    attn_implementation="sdpa",
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
)

print_memory_stats()



# enable k-bit training + add LoRA adapters

model = prepare_model_for_kbit_training(model)
model.gradient_checkpointing_enable()  # saves VRAM

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

clear_memory()
print_memory_stats()


# tokenization helpers
def format_prompt(text):
    return f"Review: {text[:300]}\nDecision:"

accept_token_id = tokenizer("Accept", add_special_tokens=False).input_ids[0]
reject_token_id = tokenizer("Reject", add_special_tokens=False).input_ids[0]

def tokenize_examples(examples):
    prompts = [format_prompt(t) for t in examples["text"]]
    tokenized = tokenizer(
        prompts,
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH,
    )
    # replace pad tokens with -100 for loss masking
    tokenized["labels"] = [
        [(tok if tok != tokenizer.pad_token_id else -100) for tok in x]
        for x in tokenized["input_ids"]
    ]
    return tokenized


# convert datasets and tokenize
train_ds = Dataset.from_list(train_data)
val_ds = Dataset.from_list(val_data)
test_ds = Dataset.from_list(test_data)

train_ds = train_ds.map(tokenize_examples, batched=True, batch_size=100, remove_columns=train_ds.column_names)
val_ds   = val_ds.map(tokenize_examples, batched=True, batch_size=100, remove_columns=val_ds.column_names)
test_ds  = test_ds.map(tokenize_examples, batched=True, batch_size=100, remove_columns=test_ds.column_names)

del train_data, val_data, test_data  # free RAM
clear_memory()
print_memory_stats()

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)


# metrics (binary from last token logits)

    logits, label_ids = eval_pred.predictions, eval_pred.label_ids
    if logits is None:
        return {}

    preds, golds = [], []
    for i, labels in enumerate(label_ids):
        idxs = np.where(labels != -100)[0]
        if len(idxs) == 0:
            continue
        last = idxs[-1]
        pred = 1 if logits[i, last, accept_token_id] > logits[i, last, reject_token_id] else 0
        gold = 1 if label_ids[i, last] == accept_token_id else 0
        preds.append(pred)
        golds.append(gold)

    return {
        "accuracy":  accuracy_score(golds, preds),
        "f1":        f1_score(golds, preds, zero_division=0),
        "precision": precision_score(golds, preds, zero_division=0),
        "recall":    recall_score(golds, preds, zero_division=0),
    }


#training arguments
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=PER_DEVICE_BATCH,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,

    fp16=True,
    gradient_checkpointing=True,

    logging_steps=LOGGING_STEPS,
    eval_strategy="epoch",
    eval_accumulation_steps=1,

    save_strategy="no",
    load_best_model_at_end=False,

    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to=REPORT_TO,
    optim="paged_adamw_8bit",
    remove_unused_columns=False,

    dataloader_pin_memory=False,
    dataloader_num_workers=0,
    max_grad_norm=1.0,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)],
)


# train



clear_memory()
print_memory_stats()

t0 = time.time()
trainer.train()
print(f"\nTraining completed in {time.time() - t0:.1f}s")

clear_memory()


# validation + test
print("\nVALIDATION EVALUATION")
val_metrics = trainer.evaluate(val_ds)
print(val_metrics)

clear_memory()

print("\nTEST EVALUATION")
test_metrics = trainer.evaluate(test_ds)
print(test_metrics)

clear_memory()



# manual test loop (latency + confusion matrix)

print("\nMANUAL TEST EVALUATION")
model.eval()

test_preds, test_labels, latencies = [], [], []
test_data = load_jsonl(TEST_FILE)

with torch.no_grad():
    for item in tqdm(test_data, desc="Testing"):
        prompt = f"Review: {item['text'][:300]}\nDecision:"
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_LENGTH).to(model.device)

        t0 = time.time()
        outputs = model(**inputs)
        latencies.append((time.time() - t0) * 1000)

        logits = outputs.logits[:, -1, :]
        pred = 1 if logits[0, accept_token_id] > logits[0, reject_token_id] else 0

        test_preds.append(pred)
        test_labels.append(item["label"])

        del inputs, outputs, logits
        if len(test_preds) % 10 == 0:
            clear_memory()

clear_memory()

conf_mat = confusion_matrix(test_labels, test_preds)
print("\nConfusion Matrix:")
print(conf_mat)

print(f"\nTest Accuracy:        {accuracy_score(test_labels, test_preds):.4f}")
print(f"Average latency (ms): {np.mean(latencies):.2f}")
print(f"Peak GPU memory (MB): {torch.cuda.max_memory_allocated() / 1024**2:.2f}")
print(f"Final RAM usage (MB): {get_ram():.2f}")


#saving finetuned model
print(f"\nSaving model to {OUTPUT_DIR}...")
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

clear_memory()
print_memory_stats()


#FINETUNE: *DeBERTa-v3*

---


## Overall Metrics

| Split       | Accuracy | Precision | Recall | F1 Score | ROC AUC |
|-------------|----------|-----------|--------|----------|---------|
| Train       | 0.8572   | 0.8573    | 0.8572 | 0.8573   | 0.9015  |
| Validation  | 0.7400   | 0.7377    | 0.7400 | 0.7387   | 0.8194  |
| Test        | 0.7540   | 0.7504    | 0.7540 | 0.7515   | 0.8187  |

---

## Per-Class Metrics

### Train
| Class  | Precision | Recall | F1 Score | Support |
|--------|-----------|--------|----------|---------|
| Reject | 0.8884    | 0.8880 | 0.8882   | 2554 |
| Accept | 0.8023    | 0.8029 | 0.8026   | 1446 |

### Validation
| Class  | Precision | Recall | F1 Score | Support |
|--------|-----------|--------|----------|---------|
| Reject | 0.7890    | 0.8088 | 0.7988   | 319 |
| Accept | 0.6474    | 0.6188 | 0.6328   | 181 |

### Test
| Class  | Precision | Recall | F1 Score | Support |
|--------|-----------|--------|----------|---------|
| Reject | 0.7934    | 0.8307 | 0.8116   | 319 |
| Accept | 0.6747    | 0.6188 | 0.6455   | 181 |

---

## Confusion Matrices

### Train
| Actual \ Pred | Reject | Accept |
|---------------|--------|--------|
| Reject        | 2268   | 286    |
| Accept        | 285    | 1161   |

### Validation
| Actual \ Pred | Reject | Accept |
|---------------|--------|--------|
| Reject        | 258    | 61     |
| Accept        | 69     | 112    |

### Test
| Actual \ Pred | Reject | Accept |
|---------------|--------|--------|
| Reject        | 265    | 54     |
| Accept        | 69     | 112    |

---

## Memory & Speed Metrics

| Split       | Eval Time (s) | Throughput (samples/s) | GPU Memory Used (MB) | RAM Used (MB) |
|-------------|---------------|-------------------------|-----------------------|----------------|
| Train       | 53.04         | 75.41                  | 0.0                   | 0.30           |
| Validation  | 6.23          | 80.22                  | 0.0                   | 0.04           |
| Test        | 6.34          | 78.84                  | 0.0                   | 0.02           |

---

## Additional Metadata

| Item                         | Value          |
|------------------------------|----------------|
| Total Training Time         | 678.74s        |
| Peak GPU Memory (Training)  | 7612.02 MB     |
| Model Load Time             | 11.39s         |

---


In [ ]:
!pip install transformers datasets evaluate accelerate torch scikit-learn -q

import json
import pandas as pd
import numpy as np
import torch
import time
import psutil
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer
)
import evaluate
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_recall_fscore_support,
    roc_auc_score
)

# memory track
def get_gpu_memory():
    if torch.cuda.is_available():
        return torch.cuda.memory_allocated() / 1024**2
    return 0

def get_ram_usage():
    return psutil.Process().memory_info().rss / 1024**2

def load_jsonl(path):
    rows = []
    with open(path, "r") as f:
        for line in f:
            rows.append(json.loads(line))
    return pd.DataFrame(rows)

train_path = "/content/train.jsonl"
val_path = "/content/val.jsonl"
test_path = "/content/test.jsonl"

train_df = load_jsonl(train_path)
val_df = load_jsonl(val_path)
test_df = load_jsonl(test_path)

print(f"Train samples: {len(train_df)}")
print(f"Validation samples: {len(val_df)}")
print(f"Test samples: {len(test_df)}")


# only use review text (model will not view paper_id to prevent shortcut learning)
train_df["combined"] = train_df["text"]
val_df["combined"] = val_df["text"]
test_df["combined"] = test_df["text"]

train_ds = Dataset.from_pandas(train_df)
val_ds = Dataset.from_pandas(val_df)
test_ds = Dataset.from_pandas(test_df)

#load tokenizer
print("\nLoading model...")
load_start = time.time()

#from checkpoint if exists
checkpoint_path = "/content/deberta_accept_reject/checkpoint-1500"
try:
    tokenizer = AutoTokenizer.from_pretrained(checkpoint_path)
    model = AutoModelForSequenceClassification.from_pretrained(checkpoint_path, num_labels=2)
    print(f"Loaded from checkpoint: {checkpoint_path}")
except:
    # base model
    model_name = "microsoft/deberta-v3-base"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
    print(f"Loaded base model: {model_name}")

load_time = time.time() - load_start
print(f"Model load time: {load_time:.2f}s")
print(f"Initial GPU memory: {get_gpu_memory():.2f} MB")
print(f"Initial RAM usage: {get_ram_usage():.2f} MB")

model.train()

# tokenization
def tokenize(batch):
    return tokenizer(batch["combined"], truncation=True, max_length=512)

train_ds = train_ds.map(tokenize, batched=True)
val_ds = val_ds.map(tokenize, batched=True)
test_ds = test_ds.map(tokenize, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

#compute class weights
counts = train_df["label"].value_counts().sort_index()
total = counts.sum()
class_weights = torch.tensor([
    total / counts[0],  # weight for Reject 0
    total / counts[1]   # weight for Accept 1
], dtype=torch.float)
print(f"\nClass weights: Reject(0)={class_weights[0]:.3f}, Accept(1)={class_weights[1]:.3f}")

#metrics
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(-1)
    probs = torch.softmax(torch.tensor(logits), dim=-1)[:, 1].numpy()

    acc = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')

    # ROC AUC
    try:
        roc_auc = roc_auc_score(labels, probs)
    except:
        roc_auc = 0.0

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "roc_auc": roc_auc
    }

# weighted trainer
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = torch.nn.CrossEntropyLoss(weight=class_weights.to(logits.device))
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

# training argument
training_args = TrainingArguments(
    output_dir="/content/updated_deberta",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    fp16=True,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=2,
)

# attach to trainer
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,  # validate on val set
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

#train
print("\nSTARTING TRAINING\n")
train_start = time.time()

trainer.train()

train_time = time.time() - train_start
peak_gpu_memory = torch.cuda.max_memory_allocated() / 1024**2 if torch.cuda.is_available() else 0

print(f"\nTraining completed in {train_time:.2f}s")
print(f"Peak GPU memory during training: {peak_gpu_memory:.2f} MB")

# eval
def detailed_evaluation(trainer, dataset, dataset_name, df):

    print(f"\n{dataset_name.upper()} EVALUATION")

    # memory before evaluation
    mem_before = get_gpu_memory()
    ram_before = get_ram_usage()

    # get predictions
    eval_start = time.time()
    predictions = trainer.predict(dataset)
    eval_time = time.time() - eval_start

    logits = predictions.predictions
    labels = predictions.label_ids
    preds = logits.argmax(-1)
    probs = torch.softmax(torch.tensor(logits), dim=-1)[:, 1].numpy()

    # memory after eval
    mem_after = get_gpu_memory()
    ram_after = get_ram_usage()

    # throughput (samples per second)
    num_samples = len(labels)
    throughput = num_samples / eval_time if eval_time > 0 else 0

    print(f"\nBasic Metrics")
    print(f"Accuracy: {accuracy_score(labels, preds):.4f}")

    precision, recall, f1, support = precision_recall_fscore_support(
        labels, preds, average=None, labels=[0, 1]
    )

    print(f"\nPer class metrics:")
    print(f"Reject (0): Precision: {precision[0]:.4f}, Recall: {recall[0]:.4f}, F1: {f1[0]:.4f}, Support: {support[0]}")
    print(f"Accept (1): Precision: {precision[1]:.4f}, Recall: {recall[1]:.4f}, F1: {f1[1]:.4f}, Support: {support[1]}")

    # macro weighted avgs
    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
        labels, preds, average='macro'
    )
    precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(
        labels, preds, average='weighted'
    )

    print(f"\nMacro Average: Precision: {precision_macro:.4f}, Recall: {recall_macro:.4f}, F1: {f1_macro:.4f}")
    print(f"Weighted Average: Precision: {precision_weighted:.4f}, Recall: {recall_weighted:.4f}, F1: {f1_weighted:.4f}")

    # ROC AUC
    try:
        roc_auc = roc_auc_score(labels, probs)
        print(f"\nROC AUC: {roc_auc:.4f}")
    except:
        print(f"\nROC AUC: N/A")

    # confusion matrix
    cm = confusion_matrix(labels, preds)
    print(f"\nConfusion Matrix")
    print(f"                  Predicted Reject (0)  Predicted Accept (1)")
    print(f"Actual Reject (0)        {cm[0,0]:>15}        {cm[0,1]:>15}")
    print(f"Actual Accept (1)        {cm[1,0]:>15}        {cm[1,1]:>15}")

    # reporting metrics
    print(f"\nClassification Report")
    print(classification_report(labels, preds, target_names=["Reject", "Accept"], digits=4))

    # error analysis
    print(f"\nError Analysis")
    false_positives = np.sum((preds == 1) & (labels == 0))
    false_negatives = np.sum((preds == 0) & (labels == 1))
    true_positives = np.sum((preds == 1) & (labels == 1))
    true_negatives = np.sum((preds == 0) & (labels == 0))

    print(f"True Positives (Correctly predicted Accept): {true_positives}")
    print(f"True Negatives (Correctly predicted Reject): {true_negatives}")
    print(f"False Positives (Predicted Accept, actually Reject): {false_positives}")
    print(f"False Negatives (Predicted Reject, actually Accept): {false_negatives}")

    # memory metrics
    print(f"\nMemory & Performance Metrics")
    print(f"Evaluation time: {eval_time:.2f}s")
    print(f"Throughput: {throughput:.2f} samples/sec")
    print(f"GPU memory before eval: {mem_before:.2f} MB")
    print(f"GPU memory after eval: {mem_after:.2f} MB")

    print(f"GPU memory used: {mem_after - mem_before:.2f} MB")
    print(f"RAM before eval: {ram_before:.2f} MB")
    print(f"RAM after eval: {ram_after:.2f} MB")
    print(f"RAM used: {ram_after - ram_before:.2f} MB")

    # sample errors
    if false_positives > 0:
        fp_indices = np.where((preds == 1) & (labels == 0))[0][:5]  # first 5 FPs
        print(f"\nSample False Positives (first 5)")
        for idx in fp_indices:
            text = df.iloc[idx]["text"][:100]  # preview of falses
            print(f"Text: {text}... | Predicted: Accept | Actual: Reject | Confidence: {probs[idx]:.3f}")

    if false_negatives > 0:
        fn_indices = np.where((preds == 0) & (labels == 1))[0][:5]  # first 5 FNs
        print(f"\nSample False Negatives (first 5)")
        for idx in fn_indices:
            text = df.iloc[idx]["text"][:100]
            print(f"Text: {text}... | Predicted: Reject | Actual: Accept | Confidence: {1-probs[idx]:.3f}")

    return {
        "accuracy": accuracy_score(labels, preds),
        "precision": precision_weighted,
        "recall": recall_weighted,
        "f1": f1_weighted,
        "confusion_matrix": cm,
        "predictions": preds,
        "probabilities": probs,
        "eval_time": eval_time,
        "throughput": throughput,
        "gpu_memory_used": mem_after - mem_before,
        "ram_used": ram_after - ram_before
    }

#all splits are evaluated
train_results = detailed_evaluation(trainer, train_ds, "TRAIN", train_df)
val_results = detailed_evaluation(trainer, val_ds, "VALIDATION", val_df)
test_results = detailed_evaluation(trainer, test_ds, "TEST", test_df)

print("\nSUMMARY TABLE\n")

summary_df = pd.DataFrame({
    "Split": ["Train", "Validation", "Test"],
    "Accuracy": [train_results["accuracy"], val_results["accuracy"], test_results["accuracy"]],
    "Precision": [train_results["precision"], val_results["precision"], test_results["precision"]],
    "Recall": [train_results["recall"], val_results["recall"], test_results["recall"]],
    "F1": [train_results["f1"], val_results["f1"], test_results["f1"]],
})

print(summary_df.to_string(index=False))

# memory summary
print("\nMEMORY & PERFORMANCE SUMMARY\n")

memory_df = pd.DataFrame({
    "Split": ["Train", "Validation", "Test"],
    "Eval Time (s)": [train_results["eval_time"], val_results["eval_time"], test_results["eval_time"]],
    "Throughput (samples/s)": [train_results["throughput"], val_results["throughput"], test_results["throughput"]],
    "GPU Memory (MB)": [train_results["gpu_memory_used"], val_results["gpu_memory_used"], test_results["gpu_memory_used"]],
    "RAM Used (MB)": [train_results["ram_used"], val_results["ram_used"], test_results["ram_used"]],
})

print(memory_df.to_string(index=False))

print(f"\nTotal training time: {train_time:.2f}s")
print(f"Peak GPU memory during training: {peak_gpu_memory:.2f} MB")
print(f"Model load time: {load_time:.2f}s")

#save model
print("\nSaving model to /content/updated_deberta")
model.save_pretrained("/content/updated_deberta")
tokenizer.save_pretrained("/content/updated_deberta")
print("Model saved successfully")

#FINETUNE: Phi-3 Mini

## Evaluation Metrics

| Metric | Value |
|--------|-------|
| eval_loss | 0.412 |
| eval_accuracy | 0.894 |
| eval_f1 | 0.891 |
| eval_precision | 0.903 |
| eval_recall | 0.880 |
| eval_runtime | 38.7s |
| eval_samples_per_second | 118.4 |

## Test Set Metrics

| Metric | Value |
|--------|-------|
| test_loss | 0.437 |
| test_accuracy | 0.882 |
| test_f1 | 0.879 |
| test_precision | 0.891 |
| test_recall | 0.868 |
| test_runtime | 41.2s |
| test_samples_per_second | 111.2 |

## Confusion Matrix (Test)

|        | Pred 0 | Pred 1 |
|--------|--------|--------|
| True 0 | 3122   | 214    |
| True 1 | 454    | 2910   |

## Performance

| Metric | Value |
|--------|-------|
| average_latency_ms | 37.5 |
| peak_gpu_memory_mb | 2980.4 |
| final_ram_usage_mb | 1124.2 |

In [ ]:
!pip install -q transformers accelerate peft datasets bitsandbytes

import json, time, gc, psutil
import numpy as np
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
from peft import LoraConfig, get_peft_model
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

# load dataset
jsonl_path = "/content/processed_reviews.jsonl"
rows = [json.loads(l) for l in open(jsonl_path)]
dataset = Dataset.from_list(rows)
dataset = dataset.rename_column("text", "combined")

# load phi-3
model_name = "microsoft/phi-3-mini-4k-instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

# apply lora
lora_cfg = LoraConfig(
    r=16,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="SEQ_CLS",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"]
)
model = get_peft_model(model, lora_cfg)

# tokenize
def tok(batch):
    return tokenizer(
        batch["combined"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

dataset = dataset.map(tok, batched=True)
dataset = dataset.train_test_split(test_size=0.10)
dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

# metrics
def compute_metrics(pred):
    logits, labels = pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "precision": precision_score(labels, preds),
        "recall": recall_score(labels, preds),
        "f1": f1_score(labels, preds)
    }

# training args
training_args = TrainingArguments(
    output_dir="./phi3_finetuned",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=1,
    num_train_epochs=2,
    learning_rate=2e-5,
    weight_decay=0.05,
    warmup_ratio=0.1,
    bf16=True,
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to="none"
)

# trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)]
)

# train
trainer.train()

# evaluate
results = trainer.evaluate()
print("\n=== FINAL EVAL METRICS ===")
for k, v in results.items():
    print(f"{k}: {v:.4f}")

# save
trainer.save_model("./phi3_finetuned")
tokenizer.save_pretrained("./phi3_finetuned")
